In [1]:
import importlib

import pandas as pd

import attrition_analysis.data_selection as data_selection
import attrition_analysis.models_utils as models_utils

importlib.reload(models_utils)
importlib.reload(data_selection)

from attrition_analysis.data_selection import CATEGORICAL_MODEL_VARS, MIXED_MODEL_VARS
from attrition_analysis.models_utils import (
    split_train_test_df,
    estimators_dict,
    evaluate_thresholds_optimized_models_cv,
    mixed_models_dict_c,
    categorical_models_dict_c,
    param_distributions_dict,
    run_cross_validation_mixed,
    run_cv_gap_analysis_mixed,
    run_model_comparison_mixed,
    tune_hyperparameters_top_combinations
)

df = pd.read_csv("../../data/clean/Employee-Attrition_Clean.csv")

target = "AttritionFlag"

df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,EducationField,EmployeeCount,EmployeeNumber,Gender,...,IncomeGroup,EducationLevel,StockOption,JobLevelGroup,SatisfactionLevel,PerformanceRatingLevel,EnvironmentSatisfactionLevel,JobInvolvementLevel,RelationshipSatisfactionLevel,WorkLifeBalanceLevel
0,41,Yes,Travel_Rarely,1102,Sales,1,Life Sciences,1,1,Female,...,Medium,College,No Options,Junior Level,Very High,Excellent,Medium,High,Low,Bad
1,49,No,Travel_Frequently,279,Research & Development,8,Life Sciences,1,2,Male,...,Medium,Below College,Low,Junior Level,Medium,Outstanding,High,Medium,Very High,Better
2,37,Yes,Travel_Rarely,1373,Research & Development,2,Other,1,4,Male,...,Low,College,No Options,Entry Level,High,Excellent,Very High,Medium,Medium,Better
3,33,No,Travel_Frequently,1392,Research & Development,3,Life Sciences,1,5,Female,...,Low,Master,No Options,Entry Level,High,Excellent,Very High,High,High,Better
4,27,No,Travel_Rarely,591,Research & Development,2,Medical,1,7,Male,...,Medium,Below College,Low,Entry Level,Medium,Excellent,Low,High,Very High,Better


In [2]:
df_train, df_test = split_train_test_df(
    df=df,
    target=target,
    test_size=0.30,
    random_state=42
)

df_train.shape, df_test.shape

((1029, 43), (441, 43))

In [3]:
df_train[target].value_counts(normalize=True), df_test[target].value_counts(normalize=True)

(AttritionFlag
 0    0.838678
 1    0.161322
 Name: proportion, dtype: float64,
 AttritionFlag
 0    0.839002
 1    0.160998
 Name: proportion, dtype: float64)

In [4]:
all_models_dict_c = {**categorical_models_dict_c, **mixed_models_dict_c}

In [5]:
cv_comparison = run_cross_validation_mixed(
    df=df_train,
    models_dict=all_models_dict_c,
    estimators_dict=estimators_dict,
    target=target,
    n_splits=10,
    random_state=42
)

cv_comparison.sort_values("F1_Mean", ascending=False).head(20)

,Variable_Set,Model,Accuracy_Mean,Accuracy_Std,Precision_Mean,Precision_Std,Recall_Mean,Recall_Std,F1_Mean,F1_Std,AUC_Mean,AUC_Std,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
141,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.783,0.038,0.410,0.061,0.758,0.107,0.531,0.071,0.847,0.069,7,11,43
140,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.880,0.022,0.738,0.138,0.422,0.112,0.526,0.105,0.856,0.067,7,11,43
147,Modelo 8 — Integrado Multidimensional,Gradient Boosting Balanced,0.833,0.034,0.499,0.090,0.571,0.126,0.522,0.074,0.804,0.068,7,11,43
149,Modelo 8 — Integrado Multidimensional,XGBoost Balanced,0.825,0.038,0.483,0.128,0.566,0.122,0.511,0.091,0.807,0.064,7,11,43
145,Modelo 8 — Integrado Multidimensional,Random Forest Balanced,0.815,0.032,0.445,0.076,0.584,0.131,0.502,0.090,0.807,0.068,7,11,43
148,Modelo 8 — Integrado Multidimensional,XGBoost,0.880,0.026,0.748,0.145,0.379,0.131,0.495,0.132,0.824,0.066,7,11,43
81,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.748,0.027,0.360,0.032,0.716,0.102,0.478,0.044,0.813,0.068,3,8,25
31,Modelo 4 — Trajetória Organizacional,Logistic Regression Balanced,0.744,0.048,0.358,0.064,0.721,0.140,0.477,0.083,0.796,0.074,0,8,22
51,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.741,0.029,0.348,0.049,0.704,0.152,0.464,0.075,0.790,0.073,0,9,24
111,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.726,0.044,0.342,0.046,0.733,0.102,0.464,0.053,0.783,0.071,5,6,20


In [6]:
gap_comparison = run_cv_gap_analysis_mixed(
    df=df_train,
    models_dict=all_models_dict_c,
    estimators_dict=estimators_dict,
    target=target,
    n_splits=10,
    random_state=42
)

gap_comparison.sort_values(
    "CV_F1_Mean",
    ascending=False
).head(20)

,Variable_Set,Model,Train_F1_Mean,CV_F1_Mean,F1_Gap,Train_Recall_Mean,CV_Recall_Mean,Recall_Gap,Train_AUC_Mean,CV_AUC_Mean,AUC_Gap,Gap_Diagnosis,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
141,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.571,0.531,0.041,0.823,0.758,0.065,0.893,0.847,0.046,Stable generalization,7,11,43
140,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.606,0.526,0.080,0.483,0.422,0.062,0.890,0.856,0.033,Moderate gap,7,11,43
147,Modelo 8 — Integrado Multidimensional,Gradient Boosting Balanced,0.862,0.522,0.340,0.965,0.571,0.394,0.990,0.804,0.186,Possible overfitting,7,11,43
149,Modelo 8 — Integrado Multidimensional,XGBoost Balanced,0.862,0.511,0.351,0.968,0.566,0.402,0.991,0.807,0.184,Possible overfitting,7,11,43
145,Modelo 8 — Integrado Multidimensional,Random Forest Balanced,0.729,0.502,0.227,0.903,0.584,0.319,0.964,0.807,0.157,Possible overfitting,7,11,43
148,Modelo 8 — Integrado Multidimensional,XGBoost,0.824,0.495,0.329,0.708,0.379,0.329,0.975,0.824,0.151,Possible overfitting,7,11,43
81,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.515,0.478,0.037,0.780,0.716,0.064,0.840,0.813,0.027,Stable generalization,3,8,25
31,Modelo 4 — Trajetória Organizacional,Logistic Regression Balanced,0.507,0.477,0.030,0.762,0.721,0.041,0.827,0.796,0.031,Stable generalization,0,8,22
51,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.504,0.464,0.040,0.757,0.704,0.053,0.825,0.790,0.035,Stable generalization,0,9,24
111,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.480,0.464,0.016,0.760,0.733,0.027,0.811,0.783,0.028,Stable generalization,5,6,20


In [7]:
model_selection_table = (
    cv_comparison
    .merge(
        gap_comparison[
            [
                "Variable_Set",
                "Model",
                "Train_F1_Mean",
                "CV_F1_Mean",
                "F1_Gap",
                "Train_Recall_Mean",
                "CV_Recall_Mean",
                "Recall_Gap",
                "Train_AUC_Mean",
                "CV_AUC_Mean",
                "AUC_Gap",
                "Gap_Diagnosis"
            ]
        ],
        on=["Variable_Set", "Model"],
        how="inner"
    )
)

model_selection_table.sort_values(
    "F1_Mean",
    ascending=False
).head(20)

,Variable_Set,Model,Accuracy_Mean,Accuracy_Std,Precision_Mean,Precision_Std,Recall_Mean,Recall_Std,F1_Mean,F1_Std,...,Train_F1_Mean,CV_F1_Mean,F1_Gap,Train_Recall_Mean,CV_Recall_Mean,Recall_Gap,Train_AUC_Mean,CV_AUC_Mean,AUC_Gap,Gap_Diagnosis
141,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.783,0.038,0.410,0.061,0.758,0.107,0.531,0.071,...,0.571,0.531,0.041,0.823,0.758,0.065,0.893,0.847,0.046,Stable generalization
140,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.880,0.022,0.738,0.138,0.422,0.112,0.526,0.105,...,0.606,0.526,0.080,0.483,0.422,0.062,0.890,0.856,0.033,Moderate gap
147,Modelo 8 — Integrado Multidimensional,Gradient Boosting Balanced,0.833,0.034,0.499,0.090,0.571,0.126,0.522,0.074,...,0.862,0.522,0.340,0.965,0.571,0.394,0.990,0.804,0.186,Possible overfitting
149,Modelo 8 — Integrado Multidimensional,XGBoost Balanced,0.825,0.038,0.483,0.128,0.566,0.122,0.511,0.091,...,0.862,0.511,0.351,0.968,0.566,0.402,0.991,0.807,0.184,Possible overfitting
145,Modelo 8 — Integrado Multidimensional,Random Forest Balanced,0.815,0.032,0.445,0.076,0.584,0.131,0.502,0.090,...,0.729,0.502,0.227,0.903,0.584,0.319,0.964,0.807,0.157,Possible overfitting
148,Modelo 8 — Integrado Multidimensional,XGBoost,0.880,0.026,0.748,0.145,0.379,0.131,0.495,0.132,...,0.824,0.495,0.329,0.708,0.379,0.329,0.975,0.824,0.151,Possible overfitting
81,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.748,0.027,0.360,0.032,0.716,0.102,0.478,0.044,...,0.515,0.478,0.037,0.780,0.716,0.064,0.840,0.813,0.027,Stable generalization
31,Modelo 4 — Trajetória Organizacional,Logistic Regression Balanced,0.744,0.048,0.358,0.064,0.721,0.140,0.477,0.083,...,0.507,0.477,0.030,0.762,0.721,0.041,0.827,0.796,0.031,Stable generalization
51,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.741,0.029,0.348,0.049,0.704,0.152,0.464,0.075,...,0.504,0.464,0.040,0.757,0.704,0.053,0.825,0.790,0.035,Stable generalization
111,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.726,0.044,0.342,0.046,0.733,0.102,0.464,0.053,...,0.480,0.464,0.016,0.760,0.733,0.027,0.811,0.783,0.028,Stable generalization


In [8]:
stable_model_selection = model_selection_table[
    (model_selection_table["F1_Gap"].abs() <= 0.10)
    & (model_selection_table["CV_F1_Mean"] >= 0.40)
].copy()

stable_model_selection = stable_model_selection.sort_values(
    ["F1_Mean", "AUC_Mean", "F1_Gap"],
    ascending=[False, False, True]
)

stable_model_selection.head(20)

,Variable_Set,Model,Accuracy_Mean,Accuracy_Std,Precision_Mean,Precision_Std,Recall_Mean,Recall_Std,F1_Mean,F1_Std,...,Train_F1_Mean,CV_F1_Mean,F1_Gap,Train_Recall_Mean,CV_Recall_Mean,Recall_Gap,Train_AUC_Mean,CV_AUC_Mean,AUC_Gap,Gap_Diagnosis
141,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.783,0.038,0.410,0.061,0.758,0.107,0.531,0.071,...,0.571,0.531,0.041,0.823,0.758,0.065,0.893,0.847,0.046,Stable generalization
140,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.880,0.022,0.738,0.138,0.422,0.112,0.526,0.105,...,0.606,0.526,0.080,0.483,0.422,0.062,0.890,0.856,0.033,Moderate gap
81,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.748,0.027,0.360,0.032,0.716,0.102,0.478,0.044,...,0.515,0.478,0.037,0.780,0.716,0.064,0.840,0.813,0.027,Stable generalization
31,Modelo 4 — Trajetória Organizacional,Logistic Regression Balanced,0.744,0.048,0.358,0.064,0.721,0.140,0.477,0.083,...,0.507,0.477,0.030,0.762,0.721,0.041,0.827,0.796,0.031,Stable generalization
51,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.741,0.029,0.348,0.049,0.704,0.152,0.464,0.075,...,0.504,0.464,0.040,0.757,0.704,0.053,0.825,0.790,0.035,Stable generalization
111,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.726,0.044,0.342,0.046,0.733,0.102,0.464,0.053,...,0.480,0.464,0.016,0.760,0.733,0.027,0.811,0.783,0.028,Stable generalization
41,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.734,0.054,0.347,0.057,0.692,0.100,0.459,0.059,...,0.508,0.459,0.049,0.762,0.692,0.070,0.832,0.795,0.037,Stable generalization
101,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.729,0.042,0.338,0.065,0.721,0.163,0.459,0.092,...,0.492,0.459,0.034,0.765,0.721,0.044,0.814,0.791,0.023,Stable generalization
121,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.735,0.028,0.341,0.043,0.703,0.132,0.458,0.066,...,0.500,0.458,0.041,0.756,0.703,0.053,0.828,0.791,0.037,Stable generalization
1,Modelo 1 — Função Profissional,Logistic Regression Balanced,0.726,0.058,0.339,0.077,0.697,0.109,0.455,0.089,...,0.482,0.455,0.027,0.740,0.697,0.043,0.831,0.789,0.042,Stable generalization


In [9]:
top_combinations = (
    stable_model_selection
    .head(20)
    [["Variable_Set", "Model"]]
    .to_dict("records")
)

top_combinations

[{'Variable_Set': 'Modelo 8 — Integrado Multidimensional',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 8 — Integrado Multidimensional',
  'Model': 'Logistic Regression'},
 {'Variable_Set': 'Modelo 2 — Nível Hierárquico e Benefícios',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 4 — Trajetória Organizacional',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 6 — Perfil Pessoal',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 5 — Antiguidade Organizacional',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 5 — Estabilidade e Benefícios',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 4 — Experiência Profissional',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 6 — Perfil Pessoal e Condições de Trabalho',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 1 — Função Profissional',
  'Model': 'Logistic Regressio

In [10]:
tuning_results_df, best_models = tune_hyperparameters_top_combinations(
    df=df_train,
    models_dict=all_models_dict_c,
    estimators_dict=estimators_dict,
    param_distributions_dict=param_distributions_dict,
    top_combinations=top_combinations,
    target=target,
    n_iter=30,
    n_splits=10,
    scoring="f1",
    random_state=42
)

tuning_results_df.sort_values(
    "Best_Score",
    ascending=False
)

/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/model_selection/_search.py:326: UserWarning: The total space of parameters 10 is smaller than n_iter=30. Running 10 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is depr

,Variable_Set,Model,Best_Score,Scoring,Best_Params,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
1,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.542,f1,"{'classifier__solver': 'liblinear', 'classifie...",7,11,43
0,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.531,f1,"{'classifier__solver': 'liblinear', 'classifie...",7,11,43
2,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.484,f1,"{'classifier__solver': 'liblinear', 'classifie...",3,8,25
3,Modelo 4 — Trajetória Organizacional,Logistic Regression Balanced,0.482,f1,"{'classifier__solver': 'liblinear', 'classifie...",0,8,22
4,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.470,f1,"{'classifier__solver': 'liblinear', 'classifie...",0,9,24
7,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.467,f1,"{'classifier__solver': 'liblinear', 'classifie...",4,6,19
8,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.465,f1,"{'classifier__solver': 'liblinear', 'classifie...",3,8,24
6,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.465,f1,"{'classifier__solver': 'liblinear', 'classifie...",0,9,24
5,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.465,f1,"{'classifier__solver': 'liblinear', 'classifie...",5,6,20
9,Modelo 1 — Função Profissional,Logistic Regression Balanced,0.460,f1,"{'classifier__solver': 'liblinear', 'classifie...",0,8,26


In [12]:
top_20_best_models = {
    (row["Variable_Set"], row["Model"]): best_models[
        (row["Variable_Set"], row["Model"])
    ]
    for _, row in (
        tuning_results_df
        .sort_values("Best_Score", ascending=False)
        .head(20)
        .iterrows()
    )
}

top_20_best_models

{('Modelo 8 — Integrado Multidimensional',
  'Logistic Regression'): Pipeline(steps=[('preprocessor',
                  ColumnTransformer(remainder='passthrough',
                                    transformers=[('numeric_scaler',
                                                   StandardScaler(),
                                                   ['MonthlyIncome',
                                                    'TotalWorkingYears',
                                                    'YearsAtCompany',
                                                    'YearsSinceLastPromotion',
                                                    'DistanceFromHome',
                                                    'DailyRate',
                                                    'TrainingTimesLastYear'])])),
                 ('classifier',
                  LogisticRegression(C=100, max_iter=10000, penalty='l2',
                                     solver='liblinear'))]),
 ('Modelo 8 — Integrad

In [13]:
(
    threshold_cv_comparison,
    best_thresholds_cv,
    final_test_results_df,
    confusion_results_final,
    fitted_models_final
) = evaluate_thresholds_optimized_models_cv(
    df=df_train,
    df_test=df_test,
    models_dict=all_models_dict_c,
    best_models=top_20_best_models,
    estimators_dict=estimators_dict,
    target=target,
    thresholds=None,
    test_size=0.30,
    random_state=42,
    n_splits=10
)

/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penal

In [14]:
best_thresholds_cv.sort_values(
    "F1-score",
    ascending=False
)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
0,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.38,0.874,0.615,0.578,0.596,0.853
1,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.64,0.846,0.518,0.681,0.589,0.842
2,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.71,0.848,0.532,0.506,0.519,0.807
6,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.59,0.801,0.424,0.651,0.513,0.790
7,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.74,0.856,0.568,0.452,0.503,0.791
3,Modelo 4 — Trajetória Organizacional,Logistic Regression Balanced,0.57,0.791,0.408,0.651,0.501,0.792
4,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.56,0.780,0.394,0.675,0.498,0.786
10,Modelo 1 — Função Profissional Misto,Logistic Regression Balanced,0.65,0.816,0.445,0.560,0.496,0.795
5,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.61,0.801,0.419,0.608,0.496,0.785
9,Modelo 1 — Função Profissional,Logistic Regression Balanced,0.61,0.800,0.417,0.602,0.493,0.786


In [15]:
final_test_results_df.sort_values(
    "F1-score",
    ascending=False
)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
0,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.38,0.864,0.575,0.592,0.583,0.814
2,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.71,0.864,0.582,0.549,0.565,0.830
1,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.64,0.837,0.495,0.634,0.556,0.810
13,Modelo 3 — Faixa Salarial,Logistic Regression Balanced,0.58,0.810,0.437,0.634,0.517,0.797
11,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.62,0.823,0.461,0.577,0.512,0.817
5,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.61,0.821,0.456,0.577,0.509,0.796
6,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.59,0.816,0.447,0.592,0.509,0.802
8,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.60,0.807,0.430,0.606,0.503,0.781
4,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.56,0.794,0.407,0.620,0.492,0.802
12,Modelo 3 — Rendimento Quantitativo,Logistic Regression Balanced,0.65,0.819,0.447,0.535,0.487,0.796


In [16]:
final_model_comparison = (
    final_test_results_df
    .merge(
        tuning_results_df[
            [
                "Variable_Set",
                "Model",
                "Best_Score",
                "Best_Params"
            ]
        ],
        on=["Variable_Set", "Model"],
        how="left"
    )
    .merge(
        model_selection_table[
            [
                "Variable_Set",
                "Model",
                "F1_Mean",
                "F1_Std",
                "AUC_Mean",
                "F1_Gap",
                "Gap_Diagnosis"
            ]
        ],
        on=["Variable_Set", "Model"],
        how="left"
    )
)

final_model_comparison = final_model_comparison.sort_values(
    "F1-score",
    ascending=False
)

final_model_comparison

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,Best_Score,Best_Params,F1_Mean,F1_Std,AUC_Mean,F1_Gap,Gap_Diagnosis
0,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.38,0.864,0.575,0.592,0.583,0.814,0.542,"{'classifier__solver': 'liblinear', 'classifie...",0.526,0.105,0.856,0.080,Moderate gap
1,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.71,0.864,0.582,0.549,0.565,0.830,0.484,"{'classifier__solver': 'liblinear', 'classifie...",0.478,0.044,0.813,0.037,Stable generalization
2,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.64,0.837,0.495,0.634,0.556,0.810,0.531,"{'classifier__solver': 'liblinear', 'classifie...",0.531,0.071,0.847,0.041,Stable generalization
3,Modelo 3 — Faixa Salarial,Logistic Regression Balanced,0.58,0.810,0.437,0.634,0.517,0.797,0.442,"{'classifier__solver': 'liblinear', 'classifie...",0.440,0.066,0.777,0.035,Stable generalization
4,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.62,0.823,0.461,0.577,0.512,0.817,0.451,"{'classifier__solver': 'liblinear', 'classifie...",0.448,0.071,0.779,0.020,Stable generalization
5,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.61,0.821,0.456,0.577,0.509,0.796,0.467,"{'classifier__solver': 'liblinear', 'classifie...",0.459,0.092,0.791,0.034,Stable generalization
6,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.59,0.816,0.447,0.592,0.509,0.802,0.465,"{'classifier__solver': 'liblinear', 'classifie...",0.458,0.066,0.791,0.041,Stable generalization
7,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.60,0.807,0.430,0.606,0.503,0.781,0.465,"{'classifier__solver': 'liblinear', 'classifie...",0.464,0.053,0.783,0.016,Stable generalization
8,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.56,0.794,0.407,0.620,0.492,0.802,0.470,"{'classifier__solver': 'liblinear', 'classifie...",0.464,0.075,0.790,0.040,Stable generalization
9,Modelo 3 — Rendimento Quantitativo,Logistic Regression Balanced,0.65,0.819,0.447,0.535,0.487,0.796,0.451,"{'classifier__solver': 'liblinear', 'classifie...",0.444,0.077,0.779,0.027,Stable generalization


In [17]:
best_final_model = final_model_comparison.iloc[0]

best_final_model

Variable_Set                 Modelo 8 — Integrado Multidimensional
Model                                          Logistic Regression
Threshold                                                     0.38
Accuracy                                                     0.864
Precision                                                    0.575
Recall                                                       0.592
F1-score                                                     0.583
AUC                                                          0.814
Best_Score                                                   0.542
Best_Params      {'classifier__solver': 'liblinear', 'classifie...
F1_Mean                                                      0.526
F1_Std                                                       0.105
AUC_Mean                                                     0.856
F1_Gap                                                        0.08
Gap_Diagnosis                                         Moderate

In [18]:
best_model_key = (
    best_final_model["Variable_Set"],
    best_final_model["Model"]
)

best_model_key

('Modelo 8 — Integrado Multidimensional', 'Logistic Regression')

In [19]:
confusion_results_final[best_model_key]

array([[339,  31],
       [ 29,  42]])

In [20]:
fitted_models_final[best_model_key]["best_threshold"]

np.float64(0.38)

In [23]:
fitted_models_final[best_model_key]["model"]

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](43,)","['MonthlyIncome','TotalWorkingYears','YearsAtCompany',..., 'WorkLifeBalanceLevel_Best','WorkLifeBalanceLevel_Better', 'WorkLifeBalanceLevel_Good']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,43
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric_scaler', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dro

In [22]:
final_model_comparison[
    [
        "Variable_Set",
        "Model",
        "Threshold",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "AUC",
        "Best_Score",
        "F1_Mean",
        "F1_Std",
        "AUC_Mean",
        "F1_Gap",
        "Gap_Diagnosis"
    ]
].sort_values(
    "F1-score",
    ascending=False
)

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,Best_Score,F1_Mean,F1_Std,AUC_Mean,F1_Gap,Gap_Diagnosis
0,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.38,0.864,0.575,0.592,0.583,0.814,0.542,0.526,0.105,0.856,0.080,Moderate gap
1,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.71,0.864,0.582,0.549,0.565,0.830,0.484,0.478,0.044,0.813,0.037,Stable generalization
2,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.64,0.837,0.495,0.634,0.556,0.810,0.531,0.531,0.071,0.847,0.041,Stable generalization
3,Modelo 3 — Faixa Salarial,Logistic Regression Balanced,0.58,0.810,0.437,0.634,0.517,0.797,0.442,0.440,0.066,0.777,0.035,Stable generalization
4,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.62,0.823,0.461,0.577,0.512,0.817,0.451,0.448,0.071,0.779,0.020,Stable generalization
5,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.61,0.821,0.456,0.577,0.509,0.796,0.467,0.459,0.092,0.791,0.034,Stable generalization
6,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.59,0.816,0.447,0.592,0.509,0.802,0.465,0.458,0.066,0.791,0.041,Stable generalization
7,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.60,0.807,0.430,0.606,0.503,0.781,0.465,0.464,0.053,0.783,0.016,Stable generalization
8,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.56,0.794,0.407,0.620,0.492,0.802,0.470,0.464,0.075,0.790,0.040,Stable generalization
9,Modelo 3 — Rendimento Quantitativo,Logistic Regression Balanced,0.65,0.819,0.447,0.535,0.487,0.796,0.451,0.444,0.077,0.779,0.027,Stable generalization
